# Chicago Public Library Coverage: One End-to-End Pipeline

This notebook defines and submits **a single Kubeflow Pipeline** that takes raw
public data about Chicago's libraries and neighborhoods all the way to a
trained, registered machine-learning model. Cleaning, travel-time mapping,
coverage scoring and modeling are each one task in a single pipeline graph,
started with one click.

**How to use it:** run the cells top to bottom. Nothing heavy happens in this
workbench. The cells only *describe* the work. The last cell compiles that
description and hands it to the cluster's pipeline server, which runs each step
in its own container. Follow the run under **Data Science Pipelines → Runs** in
the OpenShift AI dashboard.

**Contents**

1. What a pipeline is, and why we use one
2. The Kubeflow Pipelines (KFP) SDK in five ideas
3. The CPL coverage example: question, data, and outputs
4. Setup
5. Step 1: Clean the raw data (three parallel tasks)
6. Step 2: Fetch travel-time isochrones (three chained tasks)
7. Step 3: Compute tract-by-branch coverage
8. Step 4: Aggregate tract-level scores
9. Step 5: Train the coverage model and predict a new branch
10. Wire the graph
11. Compile and submit

## 1. What is a pipeline, and why use one?

A **machine-learning pipeline** is a workflow broken into discrete steps, with
the dependencies between them written down explicitly. Picture a flowchart:
each box is a step ("clean the data", "train the model"), and each arrow means
"this step needs the previous one to finish first". Formally, that's a
*directed acyclic graph* (DAG).

Writing the work as a pipeline, rather than a notebook you run cell by cell,
gets you several things:

| Benefit | What it means in practice |
|---|---|
| **Reproducibility** | Each run records its exact code, parameters, container image and inputs. You can rerun last Tuesday's model exactly. |
| **Isolation** | Each step runs in its own container with its own libraries. A geospatial step can install `geopandas` while the training step installs `xgboost`, and neither can break the other. |
| **Traceability and lineage** | The dashboard shows which run produced which artifact, how long each step took, and its logs. |
| **Parallelism** | Steps that don't depend on each other run at the same time. |
| **Automation** | A compiled pipeline can be rerun with new parameters, put on a schedule, or triggered from CI/CD without anyone opening a notebook. |
| **Resource fit** | The workbench stays small. The heavy lifting happens in short-lived pods that disappear when they finish. |

On **Red Hat OpenShift AI**, pipelines are provided by *Data Science
Pipelines*, Red Hat's distribution of open-source **Kubeflow Pipelines**. The
Helm chart that deploys this project creates a `DataSciencePipelinesApplication`
(the pipeline server) in the `cpl-coverage` namespace. That server is what
this notebook talks to.

## 2. The Kubeflow Pipelines (KFP) SDK in five ideas

The `kfp` Python package lets you write a pipeline as ordinary Python. Five
ideas cover everything in this notebook.

**1. Components: `@dsl.component`**
A component is one step. You write a normal Python function and decorate it.
At runtime KFP starts a container from `base_image`, `pip install`s
`packages_to_install` (here from `pip_index_urls`, i.e. PyPI), then calls your
function inside that container. Because the function runs somewhere else,
**all of its imports must be inside the function body**, and it can't see
variables defined elsewhere in this notebook.

**2. Typed inputs and outputs**
Parameters such as `mode: str` or `travel_time: int` are passed by value.
`Output[Dataset]` declares a file *artifact*: KFP gives the function a local
`.path` to write to, then uploads the file to the pipeline's artifact store so
it shows up (and can be downloaded) in the dashboard.

**3. Pipelines: `@dsl.pipeline`**
The pipeline function wires components together. Calling a component inside it
**doesn't run anything**. It returns a `PipelineTask`, a node in the graph.
The pipeline function runs only once, at compile time, to build that graph.
Edges come either from passing one task's output into another, or explicitly
from `task_b.after(task_a)`.

**4. Kubernetes extensions: `kfp.kubernetes`**
Plain KFP is cluster-agnostic. The `kfp-kubernetes` extension adds
Kubernetes-specific wiring per task. We use `use_secret_as_env` (S3 and Mapbox
credentials become environment variables) and `use_config_map_as_volume` (the
Feast client config and TLS certificate become files).

**5. Compile, upload, run**
`Compiler().compile(...)` turns the Python graph into a portable YAML
definition (KFP's *intermediate representation*). `kfp.Client` uploads that
YAML to the pipeline server and starts a run. The server then schedules one pod
per task, in dependency order.

## 3. The CPL coverage example

### The question

> *How well do Chicago Public Library branches cover the city's residents on
> foot, by bike and by car, and where would a new branch help most?*

"Coverage" here means *geographic reach*: what share of each census tract (a
neighborhood-sized area of roughly 4,000 residents) lies within a short trip of
at least one open branch.

### The data

| Dataset | Source | Location in the S3 bucket | Used for |
|---|---|---|---|
| Library locations, contact info and hours | City of Chicago Data Portal | `raw/branches/…csv` | Branch names and coordinates |
| City of Chicago boundary | City of Chicago Data Portal (GeoJSON) | `raw/boundaries/Boundaries - City.geojson` | Clipping the state's tracts to the city |
| 2021 Illinois census tracts | U.S. Census Bureau TIGER/Line shapefile | `raw/boundaries/tl_2021_17_tract.zip` | Tract polygons, land and water area |
| American Community Survey tract demographics | Social Explorer export (ACS, summary level 140 = tract) | `raw/demographics/R13395758_SL140.csv` | Population, density and percentage breakdowns |
| Travel-time polygons (*isochrones*) | Mapbox Isochrone API, fetched live | `features/isochrones.parquet` (also Feast's offline store) | The area reachable from each branch |

When the project is deployed, a one-time setup job uploads these `raw/` files
into the project's S3 bucket. Everything after that is produced by this
pipeline.

### The pipeline at a glance

```
                ┌──────────────────┐
                │  clean_branches  │──┐
                └──────────────────┘  │   ┌───────────────┐   ┌───────────────┐   ┌───────────────┐
                                      └──▶│ fetch driving │──▶│ fetch cycling │──▶│ fetch walking │──┐
                                          │    (5 min)    │   │   (10 min)    │   │   (10 min)    │  │
                                          └───────────────┘   └───────────────┘   └───────────────┘  │
                ┌─────────────────────┐                                                            ▼
                │ clean_census_tracts │─────────────────────────────────────────────▶ ┌──────────────────┐
                └─────────────────────┘                                               │ compute_coverage │
                ┌─────────────────────┐                                               └────────┬─────────┘
                │ clean_census_demos  │ (side output for EDA)                                  ▼
                └─────────────────────┘                                               ┌──────────────────┐
                                                                                      │ aggregate_scores │
                                                                                      └────────┬─────────┘
                                                                                               ▼
                                                                                      ┌───────────────────┐
                                                                                      │ train_and_predict │
                                                                                      └───────────────────┘
```

| # | Task | Reads (S3) | Writes (S3) |
|---|---|---|---|
| 1a | `clean_branches` | `raw/branches/…csv` | `clean/branches.csv` |
| 1b | `clean_census_tracts` | `raw/boundaries/*`, `raw/demographics/…csv` | `clean/population_demographics.csv` |
| 1c | `clean_census_demos` | same as 1b | `clean/census_demos.csv` |
| 2 | `fetch_isochrones` ×3 | `clean/branches.csv`, Feast, Mapbox | `features/isochrones.parquet`, Feast online store |
| 3 | `compute_coverage` | `clean/population_demographics.csv`, `features/isochrones.parquet` | `results/coverage_details.parquet` |
| 4 | `aggregate_scores` | `results/coverage_details.parquet`, tracts | `results/coverage_scores.{parquet,csv}` |
| 5 | `train_and_predict` | coverage details, branches, tracts, scores | MLflow run and registered model, `models/coverage-model/model.json`, `results/coverage_predictions.{parquet,csv}` |

### How steps hand data to each other

Each task writes its result **both** as a KFP artifact (visible in the
dashboard) **and** to a fixed key in the project's S3 bucket. Downstream tasks
read from the S3 key. We use the bucket as the hand-off point because other
parts of the system read those same keys: the EDA workbench, the Feast feature
store, KServe (which serves `models/coverage-model/`) and the MCP server.
Because the hand-off goes through S3 rather than through KFP artifacts, KFP
can't infer the ordering on its own, so the graph uses explicit `.after(...)`
edges.

### Cluster resources the tasks rely on

All of these are created by the `cpl-coverage` Helm chart, which Argo CD syncs
from Git:

| Resource | Created by | Used by |
|---|---|---|
| Secret `cpl-s3` (endpoint, bucket, keys) | `data-connection.yaml` job, from the `ObjectBucketClaim` | every task |
| Secret `mapbox-token` | `mapbox-secret.yaml` | `fetch_isochrones` |
| ConfigMap `feast-client-config` | `feast-client-config.yaml` | `fetch_isochrones` |
| ConfigMap `feast-cpl-features-client-ca` | Feast operator, for the `cpl-features` FeatureStore | `fetch_isochrones` (TLS trust for both Feast and the in-cluster S3 endpoint) |
| Secret `pipeline-url` (`PIPELINES_URL`) | `pipeline-url.yaml` job | this workbench, to find the pipeline server |
| `MLFLOW_TRACKING_URI` env var | RHOAI, via the workbench's `mlflow-instance` annotation | `train_and_predict` |

## 4. Setup

`kfp` and `kfp-kubernetes` are preinstalled in the workbench image. We also
define the shared settings every task uses:

- **`RUNTIME_IMAGE`**: the Red Hat *runtime-datascience* image from the
  cluster's internal registry. It already ships `pandas`, `numpy`,
  `scikit-learn`, `pyarrow`, `boto3` and `requests`, so each step only
  downloads the few extra libraries it needs.
- **`MLFLOW_TRACKING_URI`**: read from *this workbench's* environment (RHOAI
  injects it) and passed to the training task, which runs in a separate pod
  that wouldn't otherwise know where MLflow lives.
- **Helpers** that attach Kubernetes secrets and ConfigMaps to a task, so the
  pipeline definition below stays readable.

In [ ]:
import os

import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"

# Injected into the workbench by RHOAI; forwarded to the training pod below.
MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]

S3_ENV = {
    "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
    "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
    "AWS_S3_BUCKET": "AWS_S3_BUCKET",
}


def with_s3(task):
    """Expose the project bucket's endpoint, name and credentials as env vars."""
    kubernetes.use_secret_as_env(task, secret_name="cpl-s3", secret_key_to_env=S3_ENV)
    return task


def with_mapbox_and_feast(task):
    """Add the Mapbox token, the Feast client config and the cluster's service CA."""
    kubernetes.use_secret_as_env(
        task,
        secret_name="mapbox-token",
        secret_key_to_env={"MAPBOX_TOKEN": "MAPBOX_TOKEN"},
    )
    kubernetes.use_config_map_as_volume(
        task, config_map_name="feast-client-config", mount_path="/feast-config"
    )
    kubernetes.use_config_map_as_volume(
        task,
        config_map_name="feast-cpl-features-client-ca",
        mount_path="/etc/pki/tls/custom-certs",
    )
    # The same service CA signs the in-cluster S3 endpoint, so Feast's S3 reads
    # (via s3fs/botocore) can verify TLS instead of disabling it.
    task.set_env_variable(
        name="AWS_CA_BUNDLE", value="/etc/pki/tls/custom-certs/service-ca.crt"
    )
    return task


def with_mlflow(task):
    """Point the task at the shared MLflow server, authenticating as the pod's service account."""
    task.set_env_variable(name="MLFLOW_TRACKING_URI", value=MLFLOW_TRACKING_URI)
    task.set_env_variable(name="MLFLOW_K8S_INTEGRATION", value="true")
    task.set_env_variable(name="MLFLOW_TRACKING_AUTH", value="kubernetes-namespaced")
    return task

## 5. Step 1: Clean the raw data

**Position in the pipeline:** first. These three tasks depend on nothing and
run **in parallel**. Everything downstream reads their outputs.

**Why:** raw public data is messy. Names are misspelled, coordinates are packed
into strings, the census publishes tracts for the whole state, and the
demographic export has an extra header row. Cleaning once, at the start, means
every later step works from one consistent version of the truth.

### 1a. `clean_branches`

**Dataset:** the City of Chicago library locations CSV.
**Key libraries:** `pandas` for tabular work, `boto3` for S3, and Python's
built-in `ast` for safely parsing the coordinate strings.

**Transformations, in order:**

1. **Normalize headers:** strip stray whitespace and rename `NAME` → `BRANCH`.
2. **Fix branch names** so they match the names used in the circulation and
   visitor datasets. Two "Daley" branches get disambiguating suffixes, a typo in
   *Harold Washtington* is corrected, and `King` becomes *Martin Luther King
   Jr.*
3. **Add the closed Roosevelt branch** with `PERMANENTLY_CLOSED = 1`. Historical
   data references it, so it stays in the table, but later steps filter it out
   when measuring today's coverage.
4. **Split coordinates:** the `LOCATION` column holds text like
   `"(41.87, -87.65)"`. `ast.literal_eval` turns it into a tuple, which becomes
   numeric `LATITUDE` and `LONGITUDE` columns.
5. **Sort by name** so the output is deterministic from run to run.

**Output:** `clean/branches.csv`, one row per branch.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3"],
    pip_index_urls=["https://pypi.org/simple"],
)
def clean_branches(branches_csv: Output[Dataset]):
    import ast
    import io
    import os
    import boto3
    import pandas as pd

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    raw_key = "raw/branches/Libraries_-_Locations___Contact_Information__and_Usual_Hours_of_Operation.csv"
    obj = s3.get_object(Bucket=bucket, Key=raw_key)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))

    df.columns = df.columns.str.strip()
    df = df.rename(columns={"NAME": "BRANCH"})

    name_fixes = {
        "Daley, Richard J.": "Daley, Richard J. - Bridgeport",
        "Daley, Richard M.": "Daley, Richard M. - W. Humboldt",
        "Harold Washtington Library Center": "Harold Washington Library Center",
        "King": "Martin Luther King Jr.",
    }
    df["BRANCH"] = df["BRANCH"].replace(name_fixes)
    df["PERMANENTLY_CLOSED"] = 0

    roosevelt = pd.DataFrame([{
        "BRANCH": "Roosevelt",
        "HOURS OF OPERATION": None,
        "ADDRESS": "1101 W. Taylor Street",
        "CITY": "Chicago", "STATE": "IL", "ZIP": "60607",
        "PHONE": None, "WEBSITE": None,
        "LOCATION": "(41.86943838326072, -87.6544586134924)",
        "PERMANENTLY_CLOSED": 1,
    }])
    df = pd.concat([df, roosevelt], ignore_index=True)

    coords = df["LOCATION"].apply(ast.literal_eval)
    df["LATITUDE"] = coords.apply(lambda c: float(c[0]))
    df["LONGITUDE"] = coords.apply(lambda c: float(c[1]))

    df = df.sort_values("BRANCH").reset_index(drop=True)

    df.to_csv(branches_csv.path, index=False)

    buf = io.BytesIO()
    df.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="clean/branches.csv", Body=buf.getvalue())
    print(f"Wrote {len(df)} branches to s3://{bucket}/clean/branches.csv")

### 1b. `clean_census_tracts`

**Datasets:** the city boundary (GeoJSON), the 2021 TIGER/Line Illinois tract
shapefile (zipped), and the Social Explorer ACS tract export.
**Key libraries:** `geopandas` (pandas with a geometry column and spatial
operations, built on `shapely`), and `pyogrio` (the fast file reader
`geopandas` uses for shapefiles and GeoJSON).

**Transformations, in order:**

1. **Load both geometries.** The shapefile arrives as a zip, so it's written to
   a temporary file that `geopandas` can open.
2. **Reproject** (`to_crs`) the tracts into the city boundary's coordinate
   reference system (CRS). Spatial operations only make sense when both layers
   use the same coordinate system.
3. **Clip** (`gpd.clip`) the statewide tracts to the city boundary, keeping only
   the parts of tracts that fall inside Chicago.
4. **Drop the duplicate header.** Social Explorer exports have a second,
   human-readable header row, so `iloc[1:]` removes it.
5. **Join** tracts to demographics on the 11-digit tract ID (`GEOID` = `FIPS`).
   It's a left join, so every tract survives even if demographics are missing.
6. **Select and rename:** keep the identifiers, land and water area, every
   population and population-density column, and the geometry. Then lower-case
   all column names.

**Output:** `clean/population_demographics.csv`. The geometry is stored as WKT
text (a standard text format for shapes). This is the tract table that steps 3,
4 and 5 all read.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "geopandas", "pyogrio"],
    pip_index_urls=["https://pypi.org/simple"],
)
def clean_census_tracts(tracts_csv: Output[Dataset]):
    import io
    import os
    import tempfile
    import boto3
    import pandas as pd
    import geopandas as gpd

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/Boundaries - City.geojson")
    chicago = gpd.read_file(io.BytesIO(obj["Body"].read()))

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/tl_2021_17_tract.zip")
    with tempfile.NamedTemporaryFile(suffix=".zip") as tmp:
        tmp.write(obj["Body"].read())
        tmp.flush()
        il_tracts = gpd.read_file(tmp.name)

    il_tracts = il_tracts.to_crs(chicago.crs)
    chicago_tracts = gpd.clip(il_tracts, chicago)

    obj = s3.get_object(Bucket=bucket, Key="raw/demographics/R13395758_SL140.csv")
    demos = pd.read_csv(io.BytesIO(obj["Body"].read()))
    demos = demos.iloc[1:, :]

    merged = chicago_tracts.merge(demos, how="left", left_on="GEOID", right_on="FIPS")

    id_cols = {
        "GEOID": "geoid",
        "Qualifying Name": "qualifying name",
        "ALAND": "current land area (sq. m)",
        "AWATER": "current water area (sq. m)",
    }
    merged = merged.rename(columns=id_cols)

    pop_cols = [c for c in merged.columns if c.startswith("Total Population") or
                c.startswith("Population Density") or c.startswith("Total:")]

    keep = list(id_cols.values()) + pop_cols + ["geometry"]
    keep = [c for c in keep if c in merged.columns]
    result = merged[keep].copy()
    result.columns = result.columns.str.lower()

    result.to_csv(tracts_csv.path, index=False)

    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="clean/population_demographics.csv", Body=buf.getvalue())
    print(f"Wrote {len(result)} tracts to s3://{bucket}/clean/population_demographics.csv")

### 1c. `clean_census_demos`

**Same inputs and the same clip-and-join as 1b**, but this task keeps a
different slice of the census: total population plus every `Percent…` column
(race, education, income, commute mode) and average commute time. Those columns
are converted to numbers with `pd.to_numeric(errors="coerce")`, which turns
suppressed or blank census cells into `NaN` instead of failing.

**Why it exists:** none of the later steps here read it. It feeds the EDA
workbench and the demographic overlays on the coverage heatmap, so it runs here
to keep `clean/` complete and in sync with the tract table.

**Output:** `clean/census_demos.csv`.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "geopandas", "pyogrio"],
    pip_index_urls=["https://pypi.org/simple"],
)
def clean_census_demos(demos_csv: Output[Dataset]):
    import io
    import os
    import tempfile
    import boto3
    import pandas as pd
    import geopandas as gpd

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/Boundaries - City.geojson")
    chicago = gpd.read_file(io.BytesIO(obj["Body"].read()))

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/tl_2021_17_tract.zip")
    with tempfile.NamedTemporaryFile(suffix=".zip") as tmp:
        tmp.write(obj["Body"].read())
        tmp.flush()
        il_tracts = gpd.read_file(tmp.name)

    il_tracts = il_tracts.to_crs(chicago.crs)
    chicago_tracts = gpd.clip(il_tracts, chicago)

    obj = s3.get_object(Bucket=bucket, Key="raw/demographics/R13395758_SL140.csv")
    demos = pd.read_csv(io.BytesIO(obj["Body"].read()))
    demos = demos.iloc[1:, :]

    merged = chicago_tracts.merge(demos, how="left", left_on="GEOID", right_on="FIPS")

    keep_cols = ["GEOID", "Qualifying Name", "geometry", "Total Population"]
    pct_cols = [c for c in merged.columns if c.startswith("Percent") or c.startswith("average commute")]
    keep_cols.extend(pct_cols)
    keep_cols = [c for c in keep_cols if c in merged.columns]
    result = merged[keep_cols].copy()

    result = result.rename(columns={
        "GEOID": "geoid",
        "Qualifying Name": "qualifying name",
        "Total Population": "total population",
    })

    for col in result.columns:
        if col.startswith("Percent") or col.startswith("average"):
            result[col] = pd.to_numeric(result[col], errors="coerce")

    result.to_csv(demos_csv.path, index=False)

    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="clean/census_demos.csv", Body=buf.getvalue())
    print(f"Wrote {len(result)} tract demographics to s3://{bucket}/clean/census_demos.csv")

## 6. Step 2: Fetch travel-time isochrones

**Position in the pipeline:** after `clean_branches`, which supplies the branch
coordinates. One component runs **three times**: driving (5 minutes), cycling
(10 minutes) and walking (10 minutes). The three runs are **chained one after
another**, not run in parallel.

**Why:** a straight-line radius around a branch ignores how people actually
travel. Rivers, rail yards and expressways all cut across it. An **isochrone**
is the polygon of everywhere you can reach from a point within a given time by
a given mode. Each isochrone is a realistic "service area" for a branch.

**Key libraries and services:**

- **Mapbox Isochrone API**, called with `requests`. One HTTP call per branch,
  mode and time returns a GeoJSON polygon.
- **`shapely`** converts that GeoJSON into WKT text so it can be stored in a
  table.
- **Feast**, the feature store deployed by the chart as `cpl-features`. Each
  isochrone is a *feature* keyed `branch|mode|minutes`, stored with a `fetch_ts`
  timestamp. Feast keeps two copies of it:
  - the **offline store** is the full history: the parquet file
    `features/isochrones.parquet` in the project's S3 bucket. The definition
    lives in `feast/data_sources.py` (`isochrone_batch`).
  - the **online store** holds only the latest value per key, for fast
    lookups. This step uses it as a cache.
- **`s3fs`** lets Feast read the offline store from the cluster's S3 service.
  Its TLS certificate is checked against the cluster's service CA, which is
  mounted into the pod along with the Feast client config.

**What happens when the task starts:**

1. **Sync the online store from S3.** `store.materialize_incremental(...)`
   copies the newest isochrone per key from the history file into the online
   store. Feast remembers how far it has loaded, so normally this only reads
   recent rows. The Feast server keeps its data inside its pod, so if the pod
   restarts the online store comes back empty, and this call rebuilds it from
   S3 automatically.

**Then, for each open branch:**

2. **Check the cache.** `store.get_online_features(...)` looks up `fetch_ts`. If
   it's less than an hour old, the branch is skipped. Reruns therefore don't
   spend Mapbox quota, and the pipeline still picks up road-network changes over
   time.
3. **Otherwise call Mapbox**, retrying up to 3 times and pausing 0.25 seconds
   between branches.
4. **Push new polygons to the online store** (`store.push(..., PushMode.ONLINE)`)
   so the next lookup sees them immediately.
5. **Append to the history file** `features/isochrones.parquet` in S3, which is
   Feast's offline store. The file is append-only on purpose: it keeps a record
   of how each service area has changed over time. Step 3 picks the newest row
   for each combination.

**Why chained:** Mapbox allows about 300 requests per minute. One mode
(roughly 80 branches at about 4 requests per second) fits within that, but all
three at once wouldn't. `.after()` in the graph makes them run one at a time.

**Output:** `features/isochrones.parquet` and the Feast online store.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "requests", "shapely", "pandas", "pyarrow", "feast[grpcio]>=0.64,<0.66", "s3fs"],
    pip_index_urls=["https://pypi.org/simple"],
)
def fetch_isochrones(
    mode: str,
    travel_time: int,
    isochrones_out: Output[Dataset],
):
    import io
    import os
    import time
    import logging
    import requests
    import pandas as pd
    import boto3
    from datetime import datetime, timezone, timedelta
    from shapely.geometry import shape

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("fetch_isochrones")

    RATE_LIMIT_DELAY = 0.25
    MAX_RETRIES = 3
    FRESHNESS_HOURS = 1

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]
    mapbox_token = os.environ["MAPBOX_TOKEN"]

    # --- Feast setup ---
    from feast import FeatureStore
    from feast.data_source import PushMode
    
    feast_config = os.environ.get("FEAST_CONFIG_DIR", "/feast-config")
    store = FeatureStore(repo_path=feast_config)

    logger.info("Feast store initialized")

    # Feast's offline store is features/isochrones.parquet in S3. Materializing
    # copies the newest isochrone per key from it into the online store.
    # materialize_incremental remembers how far it has loaded, so normally this
    # only picks up recent rows; after a Feast restart that memory is gone and
    # the whole history (up to a year back) is reloaded.
    try:
        store.materialize_incremental(
            end_date=datetime.now(timezone.utc),
            feature_views=["isochrone_features"],
        )
        logger.info("Online store synced from the S3 history")
    except Exception as e:
        logger.warning(f"Materialization from S3 failed; using the online store as-is: {e}")

    # --- Load branches ---
    obj = s3.get_object(Bucket=bucket, Key="clean/branches.csv")
    branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
    branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
    logger.info(f"Loaded {len(branches)} active branches")

    now = datetime.now(timezone.utc)
    freshness_cutoff = now - timedelta(hours=FRESHNESS_HOURS)
    cutoff_ts = int(freshness_cutoff.timestamp())
    base_url = f"https://api.mapbox.com/isochrone/v1/mapbox/{mode}/"

    new_rows = []
    stats = {"cached": 0, "fetched": 0, "failed": 0}

    for _, branch in branches.iterrows():
        branch_name = branch["BRANCH"]
        lat, lon = branch["LATITUDE"], branch["LONGITUDE"]

        try:
            iso_key = f"{branch_name}|{mode}|{travel_time}"
            feast_result = store.get_online_features(
                features=["isochrone_features:fetch_ts"],
                entity_rows=[{
                    "isochrone_key": iso_key,
                }],
            ).to_dict()
            cached_ts = feast_result.get("fetch_ts", [None])[0]
            if cached_ts is not None and cached_ts >= cutoff_ts:
                stats["cached"] += 1
                continue
        except Exception as e:
            # warning, not debug: a silent failure means every branch refetches
            # from Mapbox while the cache appears to be working.
            logger.warning(f"Feast lookup failed for {branch_name}: {e}")

        success = False
        for attempt in range(MAX_RETRIES):
            try:
                url = (
                    f"{base_url}{lon},{lat}"
                    f"?contours_minutes={travel_time}"
                    f"&polygons=true&access_token={mapbox_token}"
                )
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                geometry = resp.json()["features"][0]["geometry"]
                geometry_wkt = shape(geometry).wkt

                new_rows.append({
                    "isochrone_key": f"{branch_name}|{mode}|{travel_time}",
                    "branch": branch_name,
                    "mode": mode,
                    "travel_time": travel_time,
                    "latitude": lat,
                    "longitude": lon,
                    "geometry_wkt": geometry_wkt,
                    "fetch_ts": int(now.timestamp()),
                    "event_timestamp": now,
                })
                stats["fetched"] += 1
                success = True
                break
            except Exception as e:
                logger.warning(
                    f"Attempt {attempt + 1}/{MAX_RETRIES} failed for "
                    f"{branch_name} ({mode}, {travel_time}min): {e}"
                )
                if attempt < MAX_RETRIES - 1:
                    time.sleep(1)

        if not success:
            stats["failed"] += 1
            logger.error(
                f"All {MAX_RETRIES} attempts failed for "
                f"{branch_name} ({mode}, {travel_time}min)"
            )

        time.sleep(RATE_LIMIT_DELAY)

    if new_rows:
        push_df = pd.DataFrame(new_rows)
        push_df["event_timestamp"] = pd.to_datetime(push_df["event_timestamp"], utc=True)
        store.push("isochrone_push", push_df, to=PushMode.ONLINE)
        logger.info(f"Pushed {len(push_df)} isochrones to Feast")

    cache_key = "features/isochrones.parquet"
    existing = pd.DataFrame()
    try:
        obj = s3.get_object(Bucket=bucket, Key=cache_key)
        existing = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    except Exception:
        pass

    if new_rows:
        new_df = pd.DataFrame(new_rows)
        result = pd.concat([existing, new_df], ignore_index=True) if not existing.empty else new_df
    else:
        result = existing if not existing.empty else pd.DataFrame()

    if not result.empty:
        buf = io.BytesIO()
        result.to_parquet(buf, index=False)
        buf.seek(0)
        s3.put_object(Bucket=bucket, Key=cache_key, Body=buf.getvalue())

    result.to_parquet(isochrones_out.path, index=False)

    logger.info(
        f"Done ({mode}, {travel_time}min). "
        f"Cached: {stats['cached']}, Fetched: {stats['fetched']}, "
        f"Failed: {stats['failed']}. Total rows: {len(result)}"
    )

## 7. Step 3: Compute tract-by-branch coverage

**Position in the pipeline:** after all three isochrone fetches *and*
`clean_census_tracts`, because it needs both the service areas and the tract
polygons.

**Why:** this step turns two map layers into numbers. For every branch
isochrone and every census tract it overlaps, it measures *how much of the
tract the branch can reach*. That single number per branch–tract pair is the
foundation for both the scores (step 4) and the training data (step 5).

**Key libraries:**

- **`shapely`**: `wkt.loads` rebuilds polygons from text, and `intersects` and
  `intersection` do the geometry.
- **`pyproj.Geod`**: measures area on the WGS84 ellipsoid (the shape of the
  Earth). The polygons are stored in latitude and longitude degrees, and a
  "square degree" isn't a fixed area. Geodesic area gives true square meters
  without picking a local map projection.
- **`geopandas`**: holds the tract table with its geometry column.

**Transformations:**

1. Load the tracts and compute each tract's true area in m².
2. Load the isochrone history and keep the **latest** polygon per
   (branch, mode, minutes) with `groupby(...).idxmax()` on the timestamp.
3. **Filter to the active combinations** (driving/5, cycling/10, walking/10).
   The history never forgets, so a retired setting (say, walking/15) would keep
   winning "latest" forever unless it's dropped explicitly.
4. For every isochrone × tract pair that touches, compute the **intersection
   polygon** and `overlap_proportion = intersection area / tract area`.

**Output:** `results/coverage_details.parquet`, one row per (tract, branch,
mode, minutes) with the overlap fraction and the intersection shape.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "shapely", "pandas", "pyarrow", "pyproj", "geopandas"],
    pip_index_urls=["https://pypi.org/simple"],
)
def compute_coverage(
    coverage_out: Output[Dataset],
):
    import io
    import os
    import logging
    import pandas as pd
    import geopandas as gpd
    import boto3
    from shapely import wkt
    from pyproj import Geod

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("compute_coverage")

    geod = Geod(ellps="WGS84")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    ACTIVE_COMBOS = {("driving", 5), ("cycling", 10), ("walking", 10)}

    # Load census tracts
    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
    tracts["geometry"] = tracts["geometry"].apply(wkt.loads)
    tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs="EPSG:4326")
    logger.info(f"Loaded {len(tracts)} census tracts")

    # Compute tract areas
    tract_areas = []
    for _, row in tracts.iterrows():
        area, _ = geod.geometry_area_perimeter(row["geometry"])
        tract_areas.append(abs(area))
    tracts["area_sqm"] = tract_areas

    # Load isochrones — use latest per (branch, mode, travel_time)
    obj = s3.get_object(Bucket=bucket, Key="features/isochrones.parquet")
    iso_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    iso_df["event_timestamp"] = pd.to_datetime(iso_df["event_timestamp"], utc=True)

    idx = iso_df.groupby(["branch", "mode", "travel_time"])["event_timestamp"].idxmax()
    iso_df = iso_df.loc[idx].reset_index(drop=True)

    # The cache is append-only and dedupes on (branch, mode, travel_time), so a
    # retired combo keeps winning idxmax forever and never ages out on its own.
    before = len(iso_df)
    iso_df = iso_df[
        [(m, int(t)) in ACTIVE_COMBOS for m, t in zip(iso_df["mode"], iso_df["travel_time"])]
    ].reset_index(drop=True)
    logger.info(f"Active-combo filter: {before} -> {len(iso_df)} isochrones")
    logger.info(
        f"Using {len(iso_df)} isochrones across "
        f"{iso_df['mode'].nunique()} modes and "
        f"{iso_df['travel_time'].nunique()} travel times"
    )

    # Compute intersections
    rows = []
    for i, iso in iso_df.iterrows():
        iso_geom = wkt.loads(iso["geometry_wkt"])
        branch = iso["branch"]
        mode = iso["mode"]
        tt = iso["travel_time"]

        for j, tract in tracts.iterrows():
            if iso_geom.intersects(tract["geometry"]):
                intersection = iso_geom.intersection(tract["geometry"])
                inter_area, _ = geod.geometry_area_perimeter(intersection)
                inter_area = abs(inter_area)
                overlap = inter_area / tract["area_sqm"] if tract["area_sqm"] > 0 else 0

                rows.append({
                    "geoid": tract["geoid"],
                    "branch": branch,
                    "mode": mode,
                    "travel_time": tt,
                    "overlap_proportion": overlap,
                    "intersection_wkt": intersection.wkt,
                })

        if (i + 1) % 25 == 0:
            logger.info(f"Processed {i + 1}/{len(iso_df)} isochrones")

    coverage = pd.DataFrame(rows)

    # Write to S3
    buf = io.BytesIO()
    coverage.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_details.parquet", Body=buf.getvalue())

    coverage.to_parquet(coverage_out.path, index=False)
    logger.info(f"Computed {len(coverage)} tract-branch intersections")

## 8. Step 4: Aggregate tract-level scores

**Position in the pipeline:** after `compute_coverage`.

**Why:** planners want one answer per neighborhood, not one per branch. This
step rolls the branch-level overlaps up into a few scores for each tract and
travel mode.

**Key libraries:** `pandas` `groupby`, `shapely.ops.unary_union`, and
`pyproj.Geod`.

**For each (tract, mode, minutes) group it computes:**

| Score | How | What it tells you |
|---|---|---|
| `raw_score` | Sum of the overlap proportions | Total "library access". Can exceed 1 where branches overlap, so it rewards redundancy. |
| `percent_covered` | Area of the **union** of all intersection shapes ÷ tract area, capped at 1 | Share of the tract within reach of **at least one** branch. Overlapping branches aren't double-counted. |
| `num_branches` | Number of rows in the group | How many branches can reach the tract. |

`unary_union` is what makes `percent_covered` honest: if two branches cover the
same block, that block counts once.

**Output:** `results/coverage_scores.parquet` (used by step 5 and the heatmap),
plus a CSV copy for quick inspection.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "shapely", "pandas", "pyarrow", "pyproj"],
    pip_index_urls=["https://pypi.org/simple"],
)
def aggregate_scores(
    aggregate_out: Output[Dataset],
):
    import io
    import os
    import logging
    import pandas as pd
    import boto3
    from shapely import wkt
    from shapely.ops import unary_union
    from pyproj import Geod

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("aggregate_scores")

    geod = Geod(ellps="WGS84")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    # Load coverage details
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_details.parquet")
    coverage = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    logger.info(f"Loaded {len(coverage)} coverage records")

    if coverage.empty:
        logger.warning("No coverage data to aggregate")
        pd.DataFrame().to_parquet(aggregate_out.path, index=False)
        return

    # Load tract geometries for area computation
    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))

    tract_area_map = {}
    for _, row in tracts.iterrows():
        geom = wkt.loads(row["geometry"])
        area, _ = geod.geometry_area_perimeter(geom)
        tract_area_map[row["geoid"]] = abs(area)

    # Aggregate per (geoid, mode, travel_time)
    results = []
    groups = coverage.groupby(["geoid", "mode", "travel_time"])
    total_groups = len(groups)

    for i, ((geoid, mode, tt), group) in enumerate(groups):
        raw_score = group["overlap_proportion"].sum()
        num_branches = len(group)

        intersection_geoms = [wkt.loads(w) for w in group["intersection_wkt"]]
        merged = unary_union(intersection_geoms)
        merged_area, _ = geod.geometry_area_perimeter(merged)
        merged_area = abs(merged_area)
        tract_area = tract_area_map.get(geoid, 1)
        percent_covered = merged_area / tract_area if tract_area > 0 else 0

        results.append({
            "geoid": geoid,
            "mode": mode,
            "travel_time": tt,
            "raw_score": raw_score,
            "percent_covered": min(percent_covered, 1.0),
            "num_branches": num_branches,
        })

        if (i + 1) % 500 == 0:
            logger.info(f"Aggregated {i + 1}/{total_groups} groups")

    result_df = pd.DataFrame(results)

    # Write parquet to S3
    buf = io.BytesIO()
    result_df.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_scores.parquet", Body=buf.getvalue())

    # Write CSV for easy viewing
    buf = io.BytesIO()
    result_df.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_scores.csv", Body=buf.getvalue())

    result_df.to_parquet(aggregate_out.path, index=False)
    logger.info(
        f"Aggregated {len(result_df)} tract scores across "
        f"{result_df['mode'].nunique()} modes. "
        f"Written to s3://{bucket}/results/coverage_scores.parquet"
    )

## 9. Step 5: Train the coverage model and predict a new branch

**Position in the pipeline:** last. It needs the per-branch coverage from
step 3 (training data) and the tract scores from step 4 (baseline for the
prediction).

**Why:** computing coverage for a *hypothetical* branch would mean a Mapbox call
and a full geometry pass for every candidate site. Instead we teach a model the
relationship between a branch's position relative to a tract and how much of
that tract it covers. Then any candidate point can be scored instantly. This is
the model KServe serves and the MCP server calls.

**Key libraries:** `xgboost` (gradient-boosted decision trees), `scikit-learn`
(train/test split and metrics), `mlflow` (experiment tracking and the model
registry), and `numpy`, `shapely` and `pyproj` for features.

**Feature engineering:** each training row is one (branch, tract, mode)
pair:

- `distance_km`: great-circle (haversine) distance from the branch to the
  tract's centroid.
- `bearing_sin`, `bearing_cos`: compass direction from branch to tract, encoded
  as sine and cosine so that 359° and 1° look close. Direction matters because
  street grids, the lake and the rivers make reach uneven.
- `tract_area_sqkm`: big tracts are harder to cover in full.
- `travel_time` plus one-hot columns `mode_driving`, `mode_cycling`,
  `mode_walking`.
- **Target:** `overlap`, the overlap proportion from step 3.

**Negative examples:** step 3 only records pairs that *do* overlap. The model
also needs to learn where coverage is zero, so for each mode we add every
uncovered branch–tract pair within **8 km** with `overlap = 0`. Pairs farther
away than that (the furthest real coverage is about 6.5 km) are trivially zero
and would only drown out the signal.

**Training and tracking:**

1. 80/20 train/test split, and an `XGBRegressor` (200 trees, depth 6, learning
   rate 0.1).
2. Log the parameters, RMSE, R² and per-feature importance to **MLflow** under
   the experiment `cpl-coverage-model`.
3. Log the model with an inferred input/output **signature** and register it as
   `cpl-coverage-model`. Features are cast to `float64` first, because MLflow's
   schema check would otherwise reject integer columns that arrive as floats at
   serving time.
4. Save a **serving copy** in XGBoost's native format to
   `models/coverage-model/model.json`. It gets its own folder because KServe
   downloads everything under the folder, and MLServer expects a native XGBoost
   file.

**Prediction:** for the pipeline parameters `new_branch_lat` and
`new_branch_lon`, predict the overlap with every tract. Add it to the tract's
current `percent_covered` (capped at 1) and report the **delta**: how much each
tract would gain. The log shows how many tracts gain more than 1% and the
population living in them.

**Output:** an MLflow run and a registered model version,
`models/coverage-model/model.json`, and `results/coverage_predictions.{parquet,csv}`.

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "pandas", "pyarrow", "numpy", "scikit-learn", "xgboost", "shapely", "pyproj", "mlflow"],
    pip_index_urls=["https://pypi.org/simple"],
)
def train_and_predict(
    new_branch_lat: float,
    new_branch_lon: float,
    mode: str = "walking",
    travel_time: int = 10,
    predictions_out: Output[Dataset] = None,
):
    import io
    import os
    import math
    import logging
    import numpy as np
    import pandas as pd
    import boto3
    import mlflow
    from mlflow.models.signature import infer_signature
    from shapely import wkt
    from pyproj import Geod
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import root_mean_squared_error, r2_score
    from xgboost import XGBRegressor

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("coverage_model")

    geod = Geod(ellps="WGS84")

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    # Furthest real coverage in the data is ~6.5km, so pairs beyond this are
    # trivially zero and only dilute the training signal.
    MAX_NEGATIVE_KM = 8.0

    mlflow.set_experiment("cpl-coverage-model")
    logger.info(f"MLflow tracking URI: {os.environ.get('MLFLOW_TRACKING_URI')}")

    # --- Helper functions ---
    def haversine_km(lat1, lon1, lat2, lon2):
        r = 6371.0
        dlat = math.radians(lat2 - lat1)
        dlon = math.radians(lon2 - lon1)
        a = math.sin(dlat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2
        return r * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    def bearing_deg(lat1, lon1, lat2, lon2):
        dlon = math.radians(lon2 - lon1)
        lat1, lat2 = math.radians(lat1), math.radians(lat2)
        x = math.sin(dlon) * math.cos(lat2)
        y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
        return (math.degrees(math.atan2(x, y)) + 360) % 360

    # --- Load data ---
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_details.parquet")
    coverage = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    logger.info(f"Loaded {len(coverage)} coverage detail records")

    obj = s3.get_object(Bucket=bucket, Key="clean/branches.csv")
    branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
    branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
    branch_coords = branches.set_index("BRANCH")[["LATITUDE", "LONGITUDE"]].to_dict("index")
    logger.info(f"Loaded {len(branches)} active branches")

    obj = s3.get_object(Bucket=bucket, Key="clean/population_demographics.csv")
    tracts = pd.read_csv(io.BytesIO(obj["Body"].read()))
    logger.info(f"Loaded {len(tracts)} census tracts")

    # Compute tract centroids and areas
    tract_info = {}
    for _, row in tracts.iterrows():
        geom = wkt.loads(row["geometry"])
        centroid = geom.centroid
        area, _ = geod.geometry_area_perimeter(geom)
        tract_info[row["geoid"]] = {
            "centroid_lat": centroid.y,
            "centroid_lon": centroid.x,
            "area_sqkm": abs(area) / 1e6,
            "total_population": row.get("total population", 0),
        }

    # --- Build training data ---
    modes_tt = coverage[["mode", "travel_time"]].drop_duplicates().values.tolist()
    all_geoids = set(tracts["geoid"])
    all_branches = set(coverage["branch"].unique())

    train_rows = []

    for m, tt in modes_tt:
        subset = coverage[(coverage["mode"] == m) & (coverage["travel_time"] == tt)]
        covered_pairs = set(zip(subset["branch"], subset["geoid"]))

        for _, row in subset.iterrows():
            b = row["branch"]
            g = row["geoid"]
            if b not in branch_coords or g not in tract_info:
                continue
            bc = branch_coords[b]
            ti = tract_info[g]
            dist = haversine_km(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
            bear = bearing_deg(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
            train_rows.append({
                "distance_km": dist,
                "bearing_sin": math.sin(math.radians(bear)),
                "bearing_cos": math.cos(math.radians(bear)),
                "tract_area_sqkm": ti["area_sqkm"],
                "travel_time": tt,
                "mode_driving": 1 if m == "driving" else 0,
                "mode_cycling": 1 if m == "cycling" else 0,
                "mode_walking": 1 if m == "walking" else 0,
                "overlap": row["overlap_proportion"],
            })

        for b in all_branches:
            if b not in branch_coords:
                continue
            bc = branch_coords[b]
            for g in all_geoids:
                if (b, g) in covered_pairs or g not in tract_info:
                    continue
                ti = tract_info[g]
                dist = haversine_km(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
                if dist > MAX_NEGATIVE_KM:
                    continue
                bear = bearing_deg(bc["LATITUDE"], bc["LONGITUDE"], ti["centroid_lat"], ti["centroid_lon"])
                train_rows.append({
                    "distance_km": dist,
                    "bearing_sin": math.sin(math.radians(bear)),
                    "bearing_cos": math.cos(math.radians(bear)),
                    "tract_area_sqkm": ti["area_sqkm"],
                    "travel_time": tt,
                    "mode_driving": 1 if m == "driving" else 0,
                    "mode_cycling": 1 if m == "cycling" else 0,
                    "mode_walking": 1 if m == "walking" else 0,
                    "overlap": 0.0,
                })

    train_df = pd.DataFrame(train_rows)
    logger.info(f"Training data: {len(train_df)} rows ({(train_df['overlap'] > 0).sum()} positive, {(train_df['overlap'] == 0).sum()} negative)")

    # --- Train model ---
    feature_cols = ["distance_km", "bearing_sin", "bearing_cos", "tract_area_sqkm",
                    "travel_time", "mode_driving", "mode_cycling", "mode_walking"]
    # float64 so the logged signature has no integer columns; MLflow schema
    # enforcement rejects ints that arrive as floats at inference time.
    X = train_df[feature_cols].astype("float64")
    y = train_df["overlap"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="reg:squarederror",
        random_state=42,
    )

    with mlflow.start_run(run_name=f"coverage-{mode}-{travel_time}min") as run:
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        rmse = root_mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        logger.info(f"Model performance — RMSE: {rmse:.4f}, R²: {r2:.4f}")

        mlflow.log_params({
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
            "mode": mode,
            "travel_time": travel_time,
            "train_rows": len(X_train),
            "test_rows": len(X_test),
            "positive_samples": int((train_df["overlap"] > 0).sum()),
            "negative_samples": int((train_df["overlap"] == 0).sum()),
            "new_branch_lat": new_branch_lat,
            "new_branch_lon": new_branch_lon,
        })

        mlflow.log_metrics({"rmse": rmse, "r2": r2})

        importance = dict(zip(feature_cols, model.feature_importances_.tolist()))
        for feat, imp in importance.items():
            mlflow.log_metric(f"importance_{feat}", imp)

        signature = infer_signature(X_test, y_pred)
        model_info = mlflow.xgboost.log_model(
            model,
            name="coverage_model",
            signature=signature,
            registered_model_name="cpl-coverage-model",
        )
        logger.info(f"MLflow run logged: {run.info.run_id}")
        logger.info(f"Model URI: {model_info.model_uri}")

    # Serving copy. KServe downloads a whole prefix and MLServer expects a
    # native xgboost file, so this needs its own directory -- a bare key
    # alongside other objects would make KServe pull them too.
    model.save_model("/tmp/model.json")
    with open("/tmp/model.json", "rb") as fh:
        s3.put_object(Bucket=bucket, Key="models/coverage-model/model.json", Body=fh.read())
    logger.info("Wrote serving copy to s3://%s/models/coverage-model/model.json", bucket)

    # --- Predict for new branch ---
    obj = s3.get_object(Bucket=bucket, Key="results/coverage_scores.parquet")
    current_scores = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    current_scores = current_scores[(current_scores["mode"] == mode) & (current_scores["travel_time"] == travel_time)]
    current_map = current_scores.set_index("geoid")["percent_covered"].to_dict()

    pred_rows = []
    for geoid, ti in tract_info.items():
        dist = haversine_km(new_branch_lat, new_branch_lon, ti["centroid_lat"], ti["centroid_lon"])
        bear = bearing_deg(new_branch_lat, new_branch_lon, ti["centroid_lat"], ti["centroid_lon"])
        pred_rows.append({
            "geoid": geoid,
            "distance_km": dist,
            "bearing_sin": math.sin(math.radians(bear)),
            "bearing_cos": math.cos(math.radians(bear)),
            "tract_area_sqkm": ti["area_sqkm"],
            "travel_time": travel_time,
            "mode_driving": 1 if mode == "driving" else 0,
            "mode_cycling": 1 if mode == "cycling" else 0,
            "mode_walking": 1 if mode == "walking" else 0,
            "total_population": ti["total_population"],
        })

    pred_df = pd.DataFrame(pred_rows)
    pred_df["predicted_overlap"] = model.predict(pred_df[feature_cols].astype("float64")).clip(0, 1)

    pred_df["current_coverage"] = pred_df["geoid"].map(current_map).fillna(0)
    pred_df["new_coverage"] = (pred_df["current_coverage"] + pred_df["predicted_overlap"]).clip(0, 1)
    pred_df["delta"] = pred_df["new_coverage"] - pred_df["current_coverage"]

    result = pred_df[["geoid", "total_population", "current_coverage", "predicted_overlap",
                       "new_coverage", "delta", "distance_km"]].copy()
    result = result.sort_values("delta", ascending=False).reset_index(drop=True)

    # Write predictions to S3
    buf = io.BytesIO()
    result.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_predictions.parquet", Body=buf.getvalue())

    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="results/coverage_predictions.csv", Body=buf.getvalue())

    result.to_parquet(predictions_out.path, index=False)

    affected = result[result["delta"] > 0.01]
    pop_affected = affected["total_population"].astype(float).sum()
    logger.info(
        f"Predicted coverage for new branch at ({new_branch_lat}, {new_branch_lon}) [{mode}/{travel_time}min]:\n"
        f"  Tracts with >1% coverage gain: {len(affected)}\n"
        f"  Population in affected tracts: {pop_affected:,.0f}\n"
        f"  Max delta: {result['delta'].max():.3f}\n"
        f"  Mean delta (affected): {affected['delta'].mean():.3f}"
    )

## 10. Wire the graph

This is the one place where the steps above become a pipeline. Things to
notice:

- **Pipeline parameters** (`new_branch_lat`, `new_branch_lon`, `mode`,
  `travel_time`) show up as editable fields when you start a run from the
  dashboard. The defaults put the candidate branch at the Loop
  (41.8781, -87.6298) and score it for a 10-minute walk.
- **`set_display_name`** gives each node a readable label in the run graph.
- **`.after(...)`** encodes the dependencies. As described in section 3, data
  moves through S3, so KFP needs to be told the order:
  - the three cleaning tasks have no `.after`, so they start together;
  - driving waits for `clean_branches`, cycling waits for driving, and walking
    waits for cycling;
  - coverage waits for walking (and therefore all fetches) *and* for the tract
    table;
  - aggregation waits for coverage, and training waits for aggregation.
- **`set_caching_options(False)`** on every task. KFP can reuse a task's
  earlier result when its inputs look identical, but these tasks read and write
  S3, Feast, Mapbox and MLflow, which KFP can't see. A "cached" step could skip
  work whose real inputs have changed, so every step always runs.

In [ ]:
@dsl.pipeline(
    name="cpl-coverage",
    description=(
        "End-to-end Chicago Public Library coverage: clean raw data, fetch "
        "isochrones, compute and aggregate tract coverage, then train the "
        "coverage model and predict the impact of a new branch."
    ),
)
def cpl_coverage_pipeline(
    new_branch_lat: float = 41.8781,
    new_branch_lon: float = -87.6298,
    mode: str = "walking",
    travel_time: int = 10,
):
    # Step 1: cleaning (parallel)
    branches = with_s3(clean_branches()).set_display_name("1a. clean branches")
    tracts = with_s3(clean_census_tracts()).set_display_name("1b. clean census tracts")
    demos = with_s3(clean_census_demos()).set_display_name("1c. clean census demographics")
    for task in (branches, tracts, demos):
        task.set_caching_options(False)

    # Step 2: isochrones (chained to respect the Mapbox rate limit)
    previous = branches
    for iso_mode, minutes in [("driving", 5), ("cycling", 10), ("walking", 10)]:
        fetch = fetch_isochrones(mode=iso_mode, travel_time=minutes)
        fetch.set_display_name(f"2. isochrones: {iso_mode} {minutes} min")
        fetch.set_caching_options(False)
        with_mapbox_and_feast(with_s3(fetch))
        fetch.after(previous)
        previous = fetch

    # Step 3: tract x branch overlap
    coverage = with_s3(compute_coverage()).set_display_name("3. compute coverage")
    coverage.set_caching_options(False)
    coverage.after(previous, tracts)

    # Step 4: tract-level scores
    scores = with_s3(aggregate_scores()).set_display_name("4. aggregate scores")
    scores.set_caching_options(False)
    scores.after(coverage)

    # Step 5: model training, registration and what-if prediction
    model = train_and_predict(
        new_branch_lat=new_branch_lat,
        new_branch_lon=new_branch_lon,
        mode=mode,
        travel_time=travel_time,
    )
    model.set_display_name("5. train model + predict new branch")
    model.set_caching_options(False)
    with_mlflow(with_s3(model))
    model.after(scores)

## 11. Compile and submit

1. **Authenticate.** `oc whoami -t` returns the OpenShift token of the user
   logged into this workbench. The pipeline server accepts it as a bearer token,
   so runs are attributed to you.
2. **Find the server.** `PIPELINES_URL` comes from the `pipeline-url` secret,
   which the chart's post-sync job fills in with the pipeline server's route.
3. **Compile** the graph to `cpl_coverage_pipeline.yaml`. That file is the
   portable artifact: it can be uploaded through the dashboard, versioned in
   Git, or submitted from CI without this notebook.
4. **Register** the definition (or a new **version** of it, if
   `cpl-coverage` already exists) so it appears under *Pipelines* and can be
   rerun or scheduled from the dashboard.
5. **Start a run of that version** in the `cpl-coverage` experiment. An
   *experiment* is a folder for related runs. There are two of them here:
   - **In the pipelines dashboard**, the cell looks the experiment up (creating
     it the first time) and submits the run by its ID, so the run can't fall
     into the server's default experiment.
   - **In MLflow**, the pipeline server records every run, with one nested MLflow
     run per task. Unless told otherwise it files them under `AIP-default`. The
     run's `plugins_input` names `cpl-coverage` instead.

   The KFP Python client in this workbench can't send `plugins_input`, so the
   run is created with one call to the pipeline server's REST API
   (`POST /apis/v2beta1/runs`), using the same token. A full run typically
   takes several minutes, mostly isochrone fetching on a cold cache and the
   geometry work in step 3.

> **Starting a run from the dashboard instead?** The dashboard doesn't know
> about this notebook's choices. On the *Create run* form, pick `cpl-coverage`
> in the **Experiment** field. Unless the form also lets you choose an MLflow
> experiment, MLflow files that run under `AIP-default`.

In [ ]:
import subprocess
import time
from datetime import datetime

import requests
import urllib3
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

PIPELINE_NAME = "cpl-coverage"
PIPELINE_FILE = "cpl_coverage_pipeline.yaml"
EXPERIMENT_NAME = "cpl-coverage"         # pipelines dashboard experiment
MLFLOW_EXPERIMENT_NAME = "cpl-coverage"  # MLflow experiment for the run

token = subprocess.check_output(["oc", "whoami", "-t"], text=True).strip()
client = kfp.Client(
    host=os.environ["PIPELINES_URL"],
    existing_token=token,
    verify_ssl=False,
)

Compiler().compile(cpl_coverage_pipeline, PIPELINE_FILE)
print(f"Compiled {PIPELINE_FILE}")

# Register the definition, or add a version if it is already there, so the
# dashboard always shows the code that was actually run.
version_name = datetime.now().strftime("v%Y%m%d-%H%M%S")
try:
    pipeline_id = client.upload_pipeline(PIPELINE_FILE, pipeline_name=PIPELINE_NAME).pipeline_id
    print(f"Registered pipeline '{PIPELINE_NAME}'.")
except ApiException as e:
    if e.status != 409:
        raise
    pipeline_id = client.get_pipeline_id(PIPELINE_NAME)
    client.upload_pipeline_version(
        PIPELINE_FILE, pipeline_version_name=version_name, pipeline_id=pipeline_id
    )
    print(f"Pipeline '{PIPELINE_NAME}' exists; added version {version_name}.")

# The version just registered is the newest one.
version_id = client.list_pipeline_versions(
    pipeline_id=pipeline_id, page_size=1, sort_by="created_at desc"
).pipeline_versions[0].pipeline_version_id

# Look up the experiment, creating it on first use, and submit by ID. Passing
# only a name lets the SDK substitute "Default" or an environment override.
experiment = client.create_experiment(
    EXPERIMENT_NAME,
    description="Runs of the end-to-end CPL library coverage pipeline",
)
print(f"Experiment: {experiment.display_name} ({experiment.experiment_id})")

# Create the run through the REST API so it can carry plugins_input, which
# tells the pipeline server's MLflow integration which experiment to use.
response = requests.post(
    f"{os.environ['PIPELINES_URL']}/apis/v2beta1/runs",
    headers={"Authorization": f"Bearer {token}"},
    json={
        "display_name": f"{PIPELINE_NAME}-{version_name}",
        "experiment_id": experiment.experiment_id,
        "pipeline_version_reference": {
            "pipeline_id": pipeline_id,
            "pipeline_version_id": version_id,
        },
        # Pipeline defaults; override e.g. {"new_branch_lat": 41.95}
        "runtime_config": {"parameters": {}},
        "plugins_input": {"mlflow": {"experiment_name": MLFLOW_EXPERIMENT_NAME}},
    },
    verify=False,
    timeout=60,
)
response.raise_for_status()
run_id = response.json()["run_id"]
print(f"Run submitted: {run_id}")
print(f"MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

Optional: block until the run finishes. You can also just watch it in the
dashboard, and stopping this cell doesn't stop the run.

In [ ]:
TERMINAL = {"SUCCEEDED", "FAILED", "SKIPPED", "ERROR", "CANCELED"}

while (state := client.get_run(run_id).state) not in TERMINAL:
    print(f"{time.strftime('%H:%M:%S')}  {state}")
    time.sleep(30)

print(f"Run finished: {state}")